# CAN Bus Anomaly Detection — Siamese DNN + Triplet Loss

Dataset: can-train-and-test (Lampe & Meng 2024) — set_01

## Pipeline
```
Parsed CSV (40-dim per-frame) → TripletDataset → Siamese DNN → Triplet Loss → Threshold
```

### 40-Dim Feature Vector
| Columns | Dims | Description |
|---------|------|-------------|
| 0–29 | 30 | One-hot CAN ID (top-30 from training normal) |
| 30 | 1 | "other" bucket for rare IDs |
| 31–38 | 8 | Data bytes / 255 (missing bytes filled -1 → -0.0039) |
| 39 | 1 | DLC / 8 |

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ── Paths (Kaggle) ──
DATA_DIR = Path("/kaggle/input/can-train-and-test")  # adjust to your uploaded dataset
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ──
TOP_K = 30
INPUT_DIM = 40
EMBEDDING_DIM = 16
HIDDEN_DIMS = [16, 32]
MARGIN = 1.0
DROPOUT = 0.3
EPOCHS = 50
BATCH_SIZE = 256
LR = 0.001
PATIENCE = 10
VAL_RATIO = 0.15
THRESHOLD_PERCENTILE = 99
SEED = 42

BYTE_COLS = [f"byte_{i}" for i in range(8)]

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed()

---
## Feature Engineering

In [ ]:
def fit_top_ids(can_ids, k=TOP_K):
    counts = pd.Series(can_ids).value_counts()
    top = counts.head(k).index.tolist()
    print(f"Top-{k} IDs: {top[:5]} ... ({len(top)} total)")
    return top


def build_features(can_ids, top_ids, byte_arr, dlc_arr):
    n = len(can_ids)
    f = np.zeros((n, 40), dtype=np.float32)
    id_map = {cid: i for i, cid in enumerate(top_ids)}
    for i, cid in enumerate(can_ids):
        f[i, id_map.get(cid, TOP_K)] = 1.0
    f[:, TOP_K + 1 : TOP_K + 9] = byte_arr
    f[:, TOP_K + 9] = dlc_arr
    return f

---
## Model Architecture
```
Input (40-dim)
  ↓
Linear(40→16) → BatchNorm → ReLU → Dropout(0.3)
  ↓
Linear(16→32) → BatchNorm → ReLU → Dropout(0.3)
  ↓
Linear(32→16) → L2 Normalize → 16-dim embedding
```

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SharedDNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, embedding_dim=EMBEDDING_DIM,
                 hidden_dims=HIDDEN_DIMS, dropout=DROPOUT):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, embedding_dim))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1)


class TripletLoss(nn.Module):
    def __init__(self, margin=MARGIN):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2) ** 2
        neg_dist = F.pairwise_distance(anchor, negative, p=2) ** 2
        return F.relu(pos_dist - neg_dist + self.margin).mean()


class SiameseNetwork(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, embedding_dim=EMBEDDING_DIM,
                 margin=MARGIN, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT):
        super().__init__()
        self.shared_dnn = SharedDNN(input_dim, embedding_dim, hidden_dims, dropout)
        self.triplet_loss = TripletLoss(margin)

    def forward(self, anchor, positive, negative):
        return self.shared_dnn(anchor), self.shared_dnn(positive), self.shared_dnn(negative)

    def get_embedding(self, x):
        return self.shared_dnn(x)

---
## Dataset & Data Loading

TripletDataset generates (anchor, positive, negative) on-the-fly:
- **Anchor**: normal frame
- **Positive**: different normal frame, same CAN ID preferred
- **Negative**: attack frame, same CAN ID preferred

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, normal_feats, normal_ids, attack_feats, attack_ids):
        self.normal = torch.tensor(normal_feats, dtype=torch.float32)
        self.attack = torch.tensor(attack_feats, dtype=torch.float32)
        self.normal_ids = normal_ids
        self.id_to_normal = self._group(normal_ids)
        self.id_to_attack = self._group(attack_ids)

    @staticmethod
    def _group(ids):
        g = {}
        for i, cid in enumerate(ids):
            g.setdefault(cid, []).append(i)
        return g

    def __len__(self):
        return len(self.normal)

    def __getitem__(self, idx):
        aid = self.normal_ids[idx]
        pos_pool = self.id_to_normal.get(aid, [])
        pos = pos_pool[np.random.randint(len(pos_pool))] if len(pos_pool) > 1 else np.random.randint(len(self.normal))
        neg_pool = self.id_to_attack.get(aid, [])
        neg = neg_pool[np.random.randint(len(neg_pool))] if neg_pool else np.random.randint(len(self.attack))
        return self.normal[idx], self.normal[pos], self.attack[neg]

In [ ]:
def chronological_split(n, val_ratio=VAL_RATIO):
    n_val = int(n * val_ratio)
    n_train = n - n_val
    return np.arange(n_train), np.arange(n_train, n)


def prepare_data(val_ratio=VAL_RATIO):
    npz_path = DATA_DIR / "train_data.npz"
    if npz_path.exists():
        print("Loading pre-built train_data.npz ...")
        d = np.load(npz_path, allow_pickle=True)
        splits = {
            "train": (d["train_feats"], d["train_ids"],
                      d["train_attack_feats"], d["train_attack_ids"],
                      d["train_attack_types"]),
            "val":   (d["val_feats"], d["val_ids"],
                      d["val_attack_feats"], d["val_attack_ids"],
                      d["val_attack_types"]),
        }
        print(f"  train: {len(d['train_feats']):,} / {len(d['train_attack_feats']):,}")
        print(f"  val:   {len(d['val_feats']):,} / {len(d['val_attack_feats']):,}")
        return splits

    df = pd.read_csv(DATA_DIR / "set01_train_frames.csv")

    n_mask = df["attack"].values == 0
    a_mask = df["attack"].values == 1

    n_ids = df.loc[n_mask, "CAN_ID"].astype(str).values
    n_ts = df.loc[n_mask, "Timestamp"].values
    n_bytes = df.loc[n_mask, BYTE_COLS].values.astype(np.float32)
    n_dlc = df.loc[n_mask, "DLC"].values.astype(np.float32)

    a_ids = df.loc[a_mask, "CAN_ID"].astype(str).values
    a_ts = df.loc[a_mask, "Timestamp"].values
    a_bytes = df.loc[a_mask, BYTE_COLS].values.astype(np.float32)
    a_dlc = df.loc[a_mask, "DLC"].values.astype(np.float32)
    a_types = df.loc[a_mask, "attack_type"].values.astype(str)

    n_order = n_ts.argsort()
    n_ids[:] = n_ids[n_order]
    n_ts[:] = n_ts[n_order]
    n_bytes[:] = n_bytes[n_order]
    n_dlc[:] = n_dlc[n_order]
    ni, nv = chronological_split(len(n_ids), val_ratio)

    ai_idx, av_idx = [], []
    for atype in np.unique(a_types):
        type_mask = a_types == atype
        type_orig_idx = np.where(type_mask)[0]
        type_order = a_ts[type_mask].argsort()
        type_orig_idx = type_orig_idx[type_order]
        ti, tv = chronological_split(len(type_orig_idx), val_ratio)
        ai_idx.extend(type_orig_idx[ti])
        av_idx.extend(type_orig_idx[tv])
    ai = np.sort(ai_idx)
    av = np.sort(av_idx)

    top_ids = fit_top_ids(n_ids[ni])
    with open(OUTPUT_DIR / "top_can_ids.json", "w") as f:
        json.dump(top_ids, f)

    n_feats = build_features(n_ids, top_ids, n_bytes, n_dlc)
    a_feats = build_features(a_ids, top_ids, a_bytes, a_dlc)

    splits = {
        "train": (n_feats[ni], n_ids[ni], a_feats[ai], a_ids[ai], a_types[ai]),
        "val":   (n_feats[nv], n_ids[nv], a_feats[av], a_ids[av], a_types[av]),
    }

    print(f"\nNormal: {len(n_ids):,} | Attack: {len(a_ids):,}")
    for name in ("train", "val"):
        nf, _, af, _, _ = splits[name]
        print(f"  {name:5s}: {len(nf):,} / {len(af):,}")

    return splits

---
## Threshold Computation

Threshold = 99th percentile of validation normal distances (computed once after training).

In [ ]:
@torch.no_grad()
def compute_threshold(model, feats, ids, device, percentile=THRESHOLD_PERCENTILE):
    model.eval()
    tensor = torch.tensor(feats, dtype=torch.float32).to(device)
    all_emb = []
    for i in tqdm(range(0, len(tensor), 256), desc="Threshold embed", leave=False):
        all_emb.append(model.shared_dnn(tensor[i:i + 256]).cpu().numpy())
    emb = np.vstack(all_emb)

    ids = np.array(ids, dtype=str)
    emb_mean = emb.mean(axis=0)
    dists = np.zeros(len(emb))
    for cid in tqdm(np.unique(ids), desc="Threshold centroids", leave=False):
        mask = ids == cid
        centroid = emb[mask].mean(axis=0)
        diff = emb[mask] - centroid[None, :]
        dists[mask] = np.sum(diff * diff, axis=1)
    unassigned = dists == 0
    if unassigned.any():
        diff = emb[unassigned] - emb_mean[None, :]
        dists[unassigned] = np.sum(diff * diff, axis=1)

    threshold = float(np.percentile(dists, percentile))
    print(f"Threshold ({percentile}th pct): {threshold:.6f}")
    return threshold

---
## Training Loop

In [ ]:
def run_epoch(model, loader, device, optimizer=None, desc=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    losses = []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    pbar = tqdm(loader, desc=desc or ("train" if is_train else "val"), leave=False)
    with ctx:
        for a, p, n in pbar:
            a, p, n = a.to(device), p.to(device), n.to(device)
            ea, ep, en = model(a, p, n)
            loss = model.triplet_loss(ea, ep, en)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.6f}")
    return np.mean(losses)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

splits = prepare_data()

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
train_loader = DataLoader(
    TripletDataset(*splits["train"][:4]), shuffle=True, drop_last=True, **loader_kw
)
val_loader = DataLoader(TripletDataset(*splits["val"][:4]), **loader_kw)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

model = SiameseNetwork(
    input_dim=INPUT_DIM, embedding_dim=EMBEDDING_DIM, margin=MARGIN, hidden_dims=HIDDEN_DIMS
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
print(f"\n{'=' * 60}")
print(f"Training for {EPOCHS} epochs (patience={PATIENCE})")
print(f"{'=' * 60}")

history = {"epoch": [], "train_loss": [], "val_loss": []}
best_val_loss = float("inf")
stale = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, device, optimizer, desc=f"E{epoch} train")
    val_loss = run_epoch(model, val_loader, device, desc=f"E{epoch} val")

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        stale = 0
        marker = " *"
        torch.save({
            "model_state_dict": model.state_dict(),
            "input_dim": INPUT_DIM,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dims": HIDDEN_DIMS,
            "margin": MARGIN,
        }, OUTPUT_DIR / "best_model.pth")
    else:
        stale += 1

    print(f"Epoch {epoch:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{marker}")

    if stale >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Compute threshold once at the end on validation normal data
print("\nComputing threshold on validation normal data ...")
threshold = compute_threshold(model, splits["val"][0], splits["val"][1], device)

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"\nDone! Best val loss: {best_val_loss:.6f} | Threshold: {threshold:.6f}")

---
## Loss Curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["epoch"], history["train_loss"], lw=2, label="Train")
ax.plot(history["epoch"], history["val_loss"], lw=2, ls="--", label="Validation")
ax.set(xlabel="Epoch", ylabel="Triplet Loss", title="Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()